In [1]:
import sys
sys.path.append("../ingestion/python/src")
sys.path.append("../ingestion/python/LangGraph_Agent")

from utils import *
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
import pandas as pd
from silver_enrichment import *
from graph_silver_enrichment import *
from APIendpoint import PlacesAPI
from database import Database

import os
from dotenv import load_dotenv
load_dotenv(override=True)

llm = LLM()
places_api = PlacesAPI(os.getenv('MAPS_APP_KEY'))
db = Database()

c:\APPS\Python312\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:

query = """
SELECT
    id_offer as job_id,
    api_source,
    job_title,
    contract_type,
    is_remote,
    offer_languages,
    seniority,
    (
        SELECT jsonb_agg(DISTINCT val)
        FROM (
            SELECT jsonb_array_elements_text(
                COALESCE(array_to_json(skills_languages)::jsonb, '[]'::jsonb) || 
                COALESCE(array_to_json(skills_frameworks)::jsonb, '[]'::jsonb) || 
                COALESCE(array_to_json(skills_aptitudes)::jsonb, '[]'::jsonb) || 
                COALESCE(array_to_json(skills_soft)::jsonb, '[]'::jsonb) ||
                COALESCE(array_to_json(alternative_job_titles)::jsonb, '[]'::jsonb)
            ) AS val
        ) sub
    ) AS all_skills,
    score_relevancy,
    explanation,
    company_name,
    website,
    primary_type,
    city,
    country,
    lat as latitude,   -- Aliased for Plotly map compatibility
    lon as longitude,  -- Aliased for Plotly map compatibility
    offer_url,
    published_at,
    collected_at
FROM serving.job_offer;
"""
db.execute('SELECT serving.refresh_job_offer_if_stale();')

result = db.execute(query)
df = pd.DataFrame(result)
df.columns.tolist()


['job_id',
 'api_source',
 'job_title',
 'contract_type',
 'is_remote',
 'offer_languages',
 'seniority',
 'all_skills',
 'score_relevancy',
 'explanation',
 'company_name',
 'website',
 'primary_type',
 'city',
 'country',
 'latitude',
 'longitude',
 'offer_url',
 'published_at',
 'collected_at']